# Sparse Matrix Computation Using CUDA — COO Format

This notebook contains the supplied **Basic Implementation** and **Detailed Implementation** converted into executable Jupyter/Google Colab cells.

The program performs Sparse Matrix–Vector Multiplication (SpMV) using **COO (Coordinate) format** and CUDA `atomicAdd()`.

For the supplied data:

```text
Matrix A =
[5  0  0  8]
[0  3  0  0]
[0  0  6  0]
[2  0  0  7]

Vector X = [1, 2, 3, 4]
```

Expected result:

```text
Y = [37, 6, 18, 30]
```


## 1. Check CUDA Environment

In Google Colab, select **Runtime → Change runtime type → GPU**.


In [ ]:
!nvidia-smi
!nvcc --version


# 2. Basic Implementation

Each CUDA thread processes one non-zero element. The operation is:

```text
product = value × X[column]
Y[row] += product
```

`atomicAdd()` is required because multiple threads can update the same output row at the same time.


In [ ]:
%%writefile sparse_basic.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define NNZ 6
#define ROWS 4

__global__ void sparseMatVec(int *row, int *col, int *val,
                             int *x, int *y);

int main()
{
    int h_row[NNZ] = {0, 0, 1, 2, 3, 3};
    int h_col[NNZ] = {0, 3, 1, 2, 0, 3};
    int h_val[NNZ] = {5, 8, 3, 6, 2, 7};

    int h_x[ROWS] = {1, 2, 3, 4};
    int h_y[ROWS] = {0};

    int *d_row, *d_col, *d_val, *d_x, *d_y;

    cudaMalloc((void**)&d_row, NNZ * sizeof(int));
    cudaMalloc((void**)&d_col, NNZ * sizeof(int));
    cudaMalloc((void**)&d_val, NNZ * sizeof(int));
    cudaMalloc((void**)&d_x, ROWS * sizeof(int));
    cudaMalloc((void**)&d_y, ROWS * sizeof(int));

    cudaMemcpy(d_row, h_row, NNZ * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_col, h_col, NNZ * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_val, h_val, NNZ * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_x, h_x, ROWS * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemset(d_y, 0, ROWS * sizeof(int));

    sparseMatVec<<<1, NNZ>>>(d_row, d_col, d_val, d_x, d_y);
    cudaDeviceSynchronize();

    cudaMemcpy(h_y, d_y, ROWS * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Output Vector:\n");
    for (int i = 0; i < ROWS; i++)
        printf("%d ", h_y[i]);
    printf("\n");

    cudaFree(d_row);
    cudaFree(d_col);
    cudaFree(d_val);
    cudaFree(d_x);
    cudaFree(d_y);

    return 0;
}

__global__ void sparseMatVec(int *row, int *col, int *val,
                             int *x, int *y)
{
    int idx = threadIdx.x;

    if (idx < NNZ)
    {
        atomicAdd(&y[row[idx]], val[idx] * x[col[idx]]);
    }
}


In [ ]:
!nvcc sparse_basic.cu -o sparse_basic
!./sparse_basic


### Expected output

```text
Output Vector:
37 6 18 30
```


# 3. Detailed Implementation

This version prints the COO elements, memory-transfer stages, kernel activity, individual products, and the final vector.

> CUDA device `printf()` output can appear in a different order because GPU threads execute concurrently.


In [ ]:
%%writefile sparse_detailed.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define NNZ 6
#define ROWS 4

__global__ void sparseMatVec(int *row, int *col, int *val,
                             int *x, int *y);

int main()
{
    // Sparse Matrix in COO Format
    int h_row[NNZ] = {0, 0, 1, 2, 3, 3};
    int h_col[NNZ] = {0, 3, 1, 2, 0, 3};
    int h_val[NNZ] = {5, 8, 3, 6, 2, 7};

    // Input Vector
    int h_x[ROWS] = {1, 2, 3, 4};

    // Output Vector
    int h_y[ROWS] = {0};

    int *d_row, *d_col, *d_val, *d_x, *d_y;

    printf("=========================================\n");
    printf("   Sparse Matrix Computation Using CUDA\n");
    printf("=========================================\n\n");

    printf("Sparse Matrix (COO Format)\n");
    printf("--------------------------\n");

    for (int i = 0; i < NNZ; i++)
    {
        printf("(%d,%d) = %d\n",
               h_row[i], h_col[i], h_val[i]);
    }

    printf("\nInput Vector:\n");
    for (int i = 0; i < ROWS; i++)
        printf("%d ", h_x[i]);
    printf("\n");

    printf("\nAllocating GPU Memory...\n");

    cudaMalloc((void**)&d_row, NNZ * sizeof(int));
    cudaMalloc((void**)&d_col, NNZ * sizeof(int));
    cudaMalloc((void**)&d_val, NNZ * sizeof(int));
    cudaMalloc((void**)&d_x, ROWS * sizeof(int));
    cudaMalloc((void**)&d_y, ROWS * sizeof(int));

    printf("GPU Memory Allocated Successfully.\n");

    printf("\nCopying Data from CPU to GPU...\n");

    cudaMemcpy(d_row, h_row, NNZ * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_col, h_col, NNZ * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_val, h_val, NNZ * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_x, h_x, ROWS * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemset(d_y, 0, ROWS * sizeof(int));

    printf("Data Copied Successfully.\n");

    printf("\nLaunching CUDA Kernel...\n\n");

    sparseMatVec<<<1, NNZ>>>(d_row, d_col, d_val, d_x, d_y);
    cudaDeviceSynchronize();

    printf("\nKernel Execution Completed.\n");

    printf("\nCopying Result from GPU to CPU...\n");

    cudaMemcpy(h_y, d_y, ROWS * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Result Copied Successfully.\n");

    printf("\nFinal Output Vector:\n");

    for (int i = 0; i < ROWS; i++)
    {
        printf("Y[%d] = %d\n", i, h_y[i]);
    }

    printf("\nReleasing GPU Memory...\n");

    cudaFree(d_row);
    cudaFree(d_col);
    cudaFree(d_val);
    cudaFree(d_x);
    cudaFree(d_y);

    printf("GPU Memory Released.\n");
    printf("\nProgram Completed Successfully.\n");

    return 0;
}

__global__ void sparseMatVec(int *row, int *col, int *val,
                             int *x, int *y)
{
    int idx = threadIdx.x;

    if (idx < NNZ)
    {
        int r = row[idx];
        int c = col[idx];
        int product = val[idx] * x[c];

        printf("Thread %d started.\n", idx);
        printf("Thread %d : Matrix[%d][%d] = %d\n",
               idx, r, c, val[idx]);
        printf("Thread %d : %d * %d = %d\n",
               idx, val[idx], x[c], product);

        atomicAdd(&y[r], product);

        printf("Thread %d : Added %d to Output[%d]\n\n",
               idx, product, r);
    }
}


In [ ]:
!nvcc sparse_detailed.cu -o sparse_detailed
!./sparse_detailed


# 4. Manual Verification

The six non-zero elements produce:

```text
A[0][0] = 5  → 5 × 1 = 5
A[0][3] = 8  → 8 × 4 = 32
A[1][1] = 3  → 3 × 2 = 6
A[2][2] = 6  → 6 × 3 = 18
A[3][0] = 2  → 2 × 1 = 2
A[3][3] = 7  → 7 × 4 = 28
```

Therefore:

```text
Y[0] = 5 + 32 = 37
Y[1] = 6
Y[2] = 18
Y[3] = 2 + 28 = 30
```

## CUDA execution

The kernel launch:

```cpp
sparseMatVec<<<1, NNZ>>>(...);
```

creates:

- 1 CUDA block
- 6 CUDA threads
- 1 thread per non-zero matrix element

For a larger matrix, use multiple blocks:

```cpp
int blockSize = 256;
int gridSize = (NNZ + blockSize - 1) / blockSize;
sparseMatVec<<<gridSize, blockSize>>>(...);
```


# 5. Laboratory Exercises

1. Change the matrix values and input vector.
2. Increase `NNZ` and add more non-zero elements.
3. Change the kernel launch to multiple blocks.
4. Add CUDA error checking using `cudaGetLastError()`.
5. Implement a CPU version and compare CPU/GPU execution.
6. Measure GPU execution time using CUDA events.
7. Implement the same SpMV operation using CSR format and compare it with COO.
